In [1]:
%load_ext autoreload
%autoreload 2

%env CUPY_ACCELERATORS=cub
%env TENSORLY_BACKEND=numpy

env: CUPY_ACCELERATORS=cub
env: TENSORLY_BACKEND=numpy


In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    AlexMI(),
    BNCI2014_001(),
    #PhysionetMI(),
    Schirrmeister2017(),
    Weibo2014(),
    Zhou2016()
]

n_classes=3
sfreq=250

In [3]:
import copy
from sklearn.base import clone
import dask
import os
import tensorly as tl
from classification_mi import stf_transform
from sklearn.preprocessing import FunctionTransformer

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)

    events = list(subj_dataset.event_id.keys())[:n_classes]
    paradigm = MotorImagery(events=events, n_classes=n_classes, resample=sfreq)

    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process(
        {pipe:clone(pipelines[pipe])},
        postprocess_pipeline= FunctionTransformer(stf_transform)
    )





In [4]:
from classification_mi import get_pipelines_mi
pipelines = get_pipelines_mi()
pipelines


{'HODA': Pipeline(steps=[('zscore1', ZScore()),
                 ('bttda',
                  BTTDACV(clf=Pipeline(steps=[('functiontransformer',
                                               FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x1487aeeb85e0>)),
                                              ('pca', PCA(whiten=True)),
                                              ('selectfcutoff', SelectFCutoff()),
                                              ('lineardiscriminantanalysis',
                                               LinearDiscriminantAnalysis(shrinkage='auto',
                                                                          solver='lsqr'))]),
                          cv=StratifiedKFold(n_splits...
                                       'tol': 0.0001, 'verbose': False},
                          max_n_blocks=1,
                          thetas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,
                                  0.9, 1.0])),
              

In [5]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from hpc import create_cluster, create_client, TIMEOUT

#import dask.config
#
#dask.config.set({
#    "distributed.scheduler.worker-saturation": 0.25
#})

import warnings
#warnings.filterwarnings("ignore", category=RuntimeWarning)
#warnings.simplefilter("error", category=UserWarning)
import pdb
%pdb on

with create_cluster(cluster='batch_sapphirerapids_full_nodes') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            n_jobs = len(dataset.subject_list)*len(pipelines)
            results += Parallel(n_jobs=n_jobs, verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Automatic pdb calling has been turned ON
Benchmarking on dataset AlexandreMotorImagery...


[Parallel(n_jobs=24)]: Using backend DaskDistributedBackend with 80 concurrent workers.
[Parallel(n_jobs=24)]: Done  15 out of  24 | elapsed:    8.7s remaining:    5.2s
[Parallel(n_jobs=24)]: Done  24 out of  24 | elapsed:    9.2s finished
[Parallel(n_jobs=27)]: Using backend DaskDistributedBackend with 1344 concurrent workers.


Benchmarking on dataset BNCI2014-001...


[Parallel(n_jobs=27)]: Done  27 out of  27 | elapsed:   18.6s finished
[Parallel(n_jobs=42)]: Using backend DaskDistributedBackend with 3840 concurrent workers.


Benchmarking on dataset Schirrmeister2017...


[Parallel(n_jobs=42)]: Done  17 out of  42 | elapsed:    7.8s remaining:   11.5s
[Parallel(n_jobs=42)]: Done  42 out of  42 | elapsed:    8.4s finished
[Parallel(n_jobs=30)]: Using backend DaskDistributedBackend with 3936 concurrent workers.


Benchmarking on dataset Weibo2014...


[Parallel(n_jobs=30)]: Done   2 out of  30 | elapsed:  5.6min remaining: 78.0min
[Parallel(n_jobs=30)]: Done  30 out of  30 | elapsed: 51.2min finished
[Parallel(n_jobs=12)]: Using backend DaskDistributedBackend with 3936 concurrent workers.


Benchmarking on dataset Zhou2016...


[Parallel(n_jobs=12)]: Done   6 out of  12 | elapsed: 12.7min remaining: 12.7min
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed: 30.4min finished


In [6]:
results.to_csv('results/moabb_mi.csv')
results

,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0.383333,18.522734,60.0,1,0,16,1,AlexandreMotorImagery,HODA
1,0.416667,51.769459,60.0,1,0,16,1,AlexandreMotorImagery,PARAFACDA
2,0.400000,91.886993,60.0,1,0,16,1,AlexandreMotorImagery,BTTDA
3,0.483333,15.685534,60.0,2,0,16,1,AlexandreMotorImagery,HODA
4,0.466667,52.661228,60.0,2,0,16,1,AlexandreMotorImagery,PARAFACDA
...,...,...,...,...,...,...,...,...,...
181,0.613333,47.115780,150.0,4,1,14,3,Zhou2016,PARAFACDA
182,0.620000,45.384293,150.0,4,2,14,3,Zhou2016,PARAFACDA
183,0.762963,134.800858,135.0,4,0,14,3,Zhou2016,BTTDA
184,0.680000,131.953522,150.0,4,1,14,3,Zhou2016,BTTDA


In [7]:
results = pd.read_csv('results/moabb_mi.csv')

In [8]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate(['mean', 'std'])

mean       std
dataset               pipeline                     
AlexandreMotorImagery BTTDA      0.495833  0.156791
                      HODA       0.500000  0.151710
                      PARAFACDA  0.466667  0.122150
BNCI2014-001          BTTDA      0.574201  0.139600
                      HODA       0.538390  0.124575
                      PARAFACDA  0.534948  0.122834
Schirrmeister2017     BTTDA      0.792409  0.125110
                      HODA       0.721843  0.090213
                      PARAFACDA  0.761144  0.132911
Weibo2014             BTTDA      0.593571  0.116119
                      HODA       0.546905  0.105346
                      PARAFACDA  0.534881  0.110431
Zhou2016              BTTDA      0.770156  0.040409
                      HODA       0.742670  0.062679
                      PARAFACDA  0.646785  0.059976

In [9]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

pipeline
BTTDA        0.645234
HODA         0.609962
PARAFACDA    0.588885
Name: score, dtype: float64

In [10]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

pipeline,subject,session,channels,n_sessions,samples,dataset,BTTDA,HODA,PARAFACDA,score_diff
0,1,0,14,3,179.0,Zhou2016,0.743016,0.703968,0.636190,0.039048
1,1,0,16,1,60.0,AlexandreMotorImagery,0.400000,0.383333,0.416667,0.016667
2,1,0,60,1,240.0,Weibo2014,0.608333,0.537500,0.491667,0.070833
3,1,0,128,1,360.0,Schirrmeister2017,0.638889,0.622222,0.575000,0.016667
4,1,0train,22,2,216.0,BNCI2014-001,0.546723,0.555285,0.610676,-0.008562
...,...,...,...,...,...,...,...,...,...,...
57,10,0,128,1,780.0,Schirrmeister2017,0.952564,0.824359,0.925641,0.128205
58,11,0,128,1,780.0,Schirrmeister2017,0.611538,0.620513,0.629487,-0.008974
59,12,0,128,1,780.0,Schirrmeister2017,0.944872,0.826923,0.920513,0.117949
60,13,0,128,1,720.0,Schirrmeister2017,0.772222,0.673611,0.719444,0.098611


In [11]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.add_shape(
        type="line",
        x0=0, y0=0.0, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig